# Coding lab — Monte Carlo inference when normal theory struggles

**Estimator sampling distributions · bias–variance–MSE · confidence-interval coverage · hypothesis testing**

**Time:** 45 minutes  
**Prerequisites:** The chapter *Estimators, Sampling Distributions, and Monte Carlo Experiments*.  
**Coding task:** Correct **four short bugs** marked `FIX ME`. Do not rewrite the simulation infrastructure. Work through the cells in order.

### Learning goals

By the end, you should be able to:

1. Construct a **finite-sample sampling distribution** by repeatedly generating samples of size $n$.
2. Estimate an estimator's **bias**, **variance**, **MSE** and the Monte Carlo uncertainty of these quantities.
3. Estimate the **coverage** of nominal 95% confidence intervals.
4. Approximate the **null sampling distribution of a test statistic** and calibrate a test using independent simulations.
5. Distinguish a **statistical approximation error** from a **Monte Carlo error**; distinguish $n$ from $B$.

> **Notation** — $n$ is the number of observations per dataset. $B$ is the number of independently simulated datasets. Our target $\mu$ is fixed, while every estimate $\bar X$ is random.

## The question: average repair cost

An insurer wants to estimate the **population mean cost per repair** (in euros). Most repairs are inexpensive; a few are costly. We model independent costs $X_i$ as **lognormal**:

$$
\log X_i \sim N(m,s^2), \qquad
m=\log(\mu)-\frac{s^2}{2}, \qquad s=1.1.
$$

The parameter of interest is $\mu=E[X_i]$. In our computer experiment we choose the *known simulation truth* $\mu=1000$ euros so that we can measure estimation errors. In actual data analysis this value would be unknown.

For this model, $\operatorname{Var}(X_i)=\mu^2(e^{s^2}-1)$, and hence

$$
E[\bar X_n]=\mu,\quad
\operatorname{Var}(\bar X_n)=\frac{\mu^2(e^{s^2}-1)}{n},\quad
\operatorname{MSE}(\bar X_n)=\operatorname{Var}(\bar X_n).
$$

These identities provide **benchmarks**, but the *exact finite-sample distribution* of the arithmetic sample mean and its studentized statistic is not a simple normal or $t$ distribution. Simulation is useful even though the model is completely specified.

**Before running anything, predict:** (a) Will $\bar X_n$ be biased? (b) For small $n$, will the sampling distribution look symmetric? (c) Must a "95%" normal- or $t$-based interval have 95% coverage here?

**Your prediction:**  
(a) …  
(b) …  
(c) …

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

# Separate seeds distinguish estimator evaluation, test calibration,
# and independent test validation.
rng_main = np.random.default_rng(2026)
rng_cal = np.random.default_rng(2027)
rng_val = np.random.default_rng(2028)

MU0 = 1000.0            # simulation truth / null mean (€)
LOG_SD = 1.1            # fixed log-scale standard deviation
ALPHA = 0.05

def draw_costs(B, n, mean_cost, rng):
    """B independent datasets, each with n observations; rows = datasets."""
    log_location = np.log(mean_cost) - LOG_SD**2 / 2
    return rng.lognormal(mean=log_location, sigma=LOG_SD, size=(B, n))

print(f"Population mean: €{MU0:,.0f}")
print(f"Population SD:   €{MU0 * np.sqrt(np.exp(LOG_SD**2) - 1):,.1f}")

## 1. The sampling distribution is a distribution of *estimates* (8 min)

Generate $B=8{,}000$ datasets, each containing $n=12$ repair costs. **Bug 1:** The scaffold takes an average along the wrong axis. Fix one line so that you get **one estimate for every dataset**.

*Hint:* The shape of `samples` is `(B, n)`. In `numpy`, `axis=1` aggregates across columns (within each row).

In [ ]:
n = 12
B = 8_000
samples = draw_costs(B=B, n=n, mean_cost=MU0, rng=rng_main)

# FIX ME (bug 1): one sample mean for EACH of the B rows.
mean_estimates = samples.mean(axis=0)

print("Samples shape:", samples.shape)
print("Estimates shape:", mean_estimates.shape)
print("Checkpoint — exactly B estimates:", mean_estimates.shape == (B,))

In [ ]:
# A histogram approximates the sampling distribution; the curve is
# the normal approximation with the correct theoretical standard error.
theory_se = MU0 * np.sqrt(np.exp(LOG_SD**2) - 1) / np.sqrt(n)
xgrid = np.linspace(0, np.quantile(mean_estimates, 0.998) * 1.05, 400)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(mean_estimates, bins=55, density=True, alpha=0.6,
        label="Monte Carlo sample means")
ax.plot(xgrid, stats.norm.pdf(xgrid, loc=MU0, scale=theory_se),
        color="black", lw=2, label="Normal approximation")
ax.axvline(MU0, color="tab:red", linestyle="--", label="True mean")
ax.set(xlabel="Sample mean (€)", ylabel="Density",
       title=f"Sampling distribution of the sample mean (n={n})")
ax.legend()
plt.show()

**Interpretation:** Is the simulated sampling distribution symmetric? Does a larger $B$ make the *estimator* more precise? Explain in two sentences: …

## 2. Bias, variance and MSE (10 min)

For independent simulated estimates $\widehat\mu^{(1)},\ldots,\widehat\mu^{(B)}$:

$$
\begin{aligned}
\widehat{\mathrm{Bias}}_{MC}&=B^{-1}\sum_b\widehat\mu^{(b)}-\mu,\\
\widehat{\mathrm{Var}}_{MC}&=B^{-1}\sum_b(\widehat\mu^{(b)}-\overline{\widehat\mu})^2,\\
\widehat{\mathrm{MSE}}_{MC}&=B^{-1}\sum_b(\widehat\mu^{(b)}-\mu)^2.
\end{aligned}
$$

With the $1/B$ convention, **exactly for the simulated values**,
$\widehat{\mathrm{MSE}}_{MC}=\widehat{\mathrm{Var}}_{MC}+\widehat{\mathrm{Bias}}_{MC}^2$.

**Bug 2:** The helper below contains *two wrong expressions*. Fix both. One confuses an estimator's expectation with its bias; the other breaks the exact Monte Carlo decomposition. You do **not** need to write a new function.

In [ ]:
def mc_summary(estimates, truth):
    estimates = np.asarray(estimates, dtype=float)
    error = estimates - truth

    # FIX ME (bug 2a): bias should measure systematic error relative to truth.
    bias = estimates.mean()

    # FIX ME (bug 2b): use the denominator convention of the displayed identity.
    variance = estimates.var(ddof=1)

    mse = np.mean(error**2)
    return {
        "bias": bias,
        "variance": variance,
        "mse": mse,
        "rmse": np.sqrt(mse),
        "mcse_bias": estimates.std(ddof=1) / np.sqrt(len(estimates)),
    }

summary = mc_summary(mean_estimates, MU0)
display(pd.Series(summary, name="Sample mean").to_frame().round(3))
gap = summary["mse"] - summary["variance"] - summary["bias"]**2
print(f"MSE - variance - bias²: {gap:.10f}")
print("Checkpoint — decomposition holds:", np.isclose(gap, 0, atol=1e-7))

In [ ]:
# Theory is available for this DGP even when exact inference is hard.
population_variance = MU0**2 * (np.exp(LOG_SD**2) - 1)
true_variance_of_mean = population_variance / n
print(f"Theoretical bias of the mean: €0")
print(f"Theoretical variance of the mean: {true_variance_of_mean:,.1f} €²")
print(f"Simulated variance: {summary['variance']:,.1f} €²")
print(f"Estimated bias: €{summary['bias']:,.1f} ± "
      f"€{1.96 * summary['mcse_bias']:,.1f} (approx. 1.96 MCSE)")

**Question:** If the estimated bias is not exactly zero, does that prove the sample mean is theoretically biased? Compare it with `mcse_bias`. Why does the variance use `ddof=0` whereas the MCSE uses `ddof=1`? …

### Optional comparison: biased versus unbiased

A **10% trimmed mean** discards the lowest and highest 10% of each sample before averaging. It is **not** generally unbiased for the population mean of skewed costs. Nevertheless, a lower variance could compensate for its bias in small samples. The code is complete; interpret the results. We use **the same simulated samples** for a fairer comparison.

In [ ]:
trim_estimates = stats.trim_mean(samples, proportiontocut=0.10, axis=1)
comparison = pd.DataFrame({
    "Arithmetic mean": mc_summary(mean_estimates, MU0),
    "10% trimmed mean": mc_summary(trim_estimates, MU0),
}).T[["bias", "variance", "mse", "rmse"]]
display(comparison.round(1))

**Question:** Which estimator has lower estimated bias? Lower variance? Lower MSE **at n = 12**? Can we conclude it will be better for all sample sizes? …

## 3. Does a nominal 95% interval actually cover 95%? (10 min)

For each sample, calculate its sample standard deviation $S_n$ and estimated standard error $S_n/\sqrt n$. Consider two familiar intervals for $\mu$:

$$
I_z=\bar X_n\pm z_{0.975}\frac{S_n}{\sqrt n},\qquad
I_t=\bar X_n\pm t_{n-1,0.975}\frac{S_n}{\sqrt n}.
$$

Both use an **estimated** standard error. Under normal observations, $I_t$ has exact coverage, but **our observations are lognormal**. Therefore neither interval is guaranteed exact at small $n$.

**Bug 3:** Correct the Boolean condition in the function `is_covered`. Coverage requires that the true value be inside **both** interval bounds.

In [ ]:
sample_se = samples.std(axis=1, ddof=1) / np.sqrt(n)
zcrit = stats.norm.ppf(1 - ALPHA / 2)
tcrit = stats.t.ppf(1 - ALPHA / 2, df=n - 1)

def is_covered(estimates, standard_errors, truth, critical_value):
    lower = estimates - critical_value * standard_errors
    upper = estimates + critical_value * standard_errors
    # FIX ME (bug 3): truth must lie between lower AND upper.
    return (lower <= truth) | (truth <= upper)

def coverage_summary(covered):
    p_hat = np.mean(covered)
    return {"coverage": p_hat,
            "mcse_coverage": np.sqrt(p_hat * (1 - p_hat) / len(covered))}

z_covered = is_covered(mean_estimates, sample_se, MU0, zcrit)
t_covered = is_covered(mean_estimates, sample_se, MU0, tcrit)
interval_table = pd.DataFrame({
    "Normal critical value": coverage_summary(z_covered),
    "t critical value": coverage_summary(t_covered),
}).T

display(interval_table.round(4))
print("Checkpoint — coverage in a plausible range:",
      0.50 < interval_table["coverage"].min() < 0.99)

**Question:** Do the intervals achieve nominal 95% coverage? Is any gap from 0.95 plausibly just Monte Carlo uncertainty? Use the reported MCSE, not just the number of decimal places. …

## 4. Hypothesis testing: simulate the null distribution (12 min)

Suppose the insurer tests

$$H_0:\ \mu=1000\quad\text{versus}\quad H_1:\ \mu\neq1000.$$

We use the familiar statistic

$$T_n=\frac{\bar X_n-\mu_0}{S_n/\sqrt n}.$$

A conventional two-sided $t$ test rejects when $|T_n|>t_{n-1,0.975}$. That calibration is **exact only for normally distributed observations**. Here we can approximate the *actual finite-sample null distribution of $T_n$* by simulation under our specified lognormal model ($s=1.1$ fixed).

For an **equal-tailed 5% test**, the Monte Carlo lower and upper critical values are the **2.5th and 97.5th percentiles** of the simulated null distribution. They need not be symmetric.

**Important:** The mean alone does not specify a null distribution. The MC-calibrated test is valid under the **fully specified lognormal null model** used here, not automatically for every population with mean €1,000.

**Bug 4:** The supplied quantiles allocate too much total rejection probability. Repair the two numbers. Keep separate calibration and validation simulations to avoid assessing a cutoff only on the data used to choose it.

In [ ]:
B_cal = 12_000
cal_samples = draw_costs(B=B_cal, n=n, mean_cost=MU0, rng=rng_cal)
cal_means = cal_samples.mean(axis=1)
cal_se = cal_samples.std(axis=1, ddof=1) / np.sqrt(n)
T_cal = (cal_means - MU0) / cal_se

# FIX ME (bug 4): equal-tail 5 percent test = 2.5 percent in each tail.
mc_low, mc_high = np.quantile(T_cal, [0.05, 0.95])

print(f"Normal-theory t critical values: {-tcrit:.3f}, {tcrit:.3f}")
print(f"Monte Carlo null critical values: {mc_low:.3f}, {mc_high:.3f}")
print("Calibration rejection rate:",
      np.mean((T_cal < mc_low) | (T_cal > mc_high)))
print("Checkpoint — close to 5 percent in calibration:",
      abs(np.mean((T_cal < mc_low) | (T_cal > mc_high)) - ALPHA) < 0.003)

In [ ]:
# Evaluate BOTH tests on FRESH datasets drawn under H0.
B_val = 12_000
val_samples = draw_costs(B=B_val, n=n, mean_cost=MU0, rng=rng_val)
val_means = val_samples.mean(axis=1)
val_se = val_samples.std(axis=1, ddof=1) / np.sqrt(n)
T_val = (val_means - MU0) / val_se

reject_t = (T_val < -tcrit) | (T_val > tcrit)
reject_mc = (T_val < mc_low) | (T_val > mc_high)

def rejection_row(indicator):
    size_hat = np.mean(indicator)
    mcse = np.sqrt(size_hat * (1 - size_hat) / len(indicator))
    return {"estimated_size": size_hat, "mcse_size": mcse}

size_results = pd.DataFrame({
    "Conventional t test": rejection_row(reject_t),
    "MC-calibrated test": rejection_row(reject_mc),
}).T
display(size_results.round(4))

# Exact logical duality: an interval excludes mu0 iff its matching test rejects.
val_t_covered = is_covered(val_means, val_se, MU0, tcrit)
print("Checkpoint — t interval/test decisions agree:",
      np.array_equal(~val_t_covered, reject_t))

In [ ]:
# The t reference distribution is symmetric. Is the simulated null distribution?
xlow, xhigh = np.quantile(T_cal, [0.005, 0.995])
xgrid_t = np.linspace(min(xlow, -7), max(xhigh, 4), 600)
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(T_cal, bins=100, density=True, alpha=0.65,
        label="MC null distribution", range=(xgrid_t[0], xgrid_t[-1]))
ax.plot(xgrid_t, stats.t.pdf(xgrid_t, df=n-1),
        color="black", lw=2, label=f"Reference t distribution (df={n-1})")
for cutoff, color, style in [(-tcrit, "tab:blue", "--"),
                              (tcrit, "tab:blue", "--"),
                              (mc_low, "tab:red", ":"),
                              (mc_high, "tab:red", ":")]:
    ax.axvline(cutoff, color=color, linestyle=style, lw=2)
ax.set(xlabel="Studentized statistic T", ylabel="Density",
       title=f"Finite-sample null distribution (n={n})")
ax.legend(loc="upper left")
plt.show()
print("Blue dashed = symmetric t cutoffs; red dotted = MC equal-tail cutoffs.")

**Question:** What are the estimated Type-I error probabilities (test sizes) for the two procedures? How does the skewed simulated null distribution explain the difference? Why use *fresh* validation samples? …

## 5. Change $n$, not $B$ (5 min)

The following code is complete. Hold $B$ fixed and vary $n$. Report the approximate MSE and coverage of the **sample mean and its $t$ interval**. The normal-reference $t$ test should improve as the CLT approximation improves, although skewness can make convergence slow.

In [ ]:
B_comparison = 8_000
rows = []
for n_this in [12, 40, 150]:
    x = draw_costs(B_comparison, n_this, MU0,
                   np.random.default_rng(4000 + n_this))
    estimates = x.mean(axis=1)
    se = x.std(axis=1, ddof=1) / np.sqrt(n_this)
    crit = stats.t.ppf(1 - ALPHA/2, df=n_this-1)
    summary_this = mc_summary(estimates, MU0)
    coverage_this = coverage_summary(is_covered(estimates, se, MU0, crit))
    rows.append({
        "n": n_this,
        "bias (€)": summary_this["bias"],
        "variance (€²)": summary_this["variance"],
        "MSE (€²)": summary_this["mse"],
        "theory MSE (€²)": population_variance / n_this,
        "t interval coverage": coverage_this["coverage"],
        "MCSE coverage": coverage_this["mcse_coverage"],
    })
display(pd.DataFrame(rows).set_index("n").round(3))

**Question:** Which changes when $n$ increases: the population, the estimator's sampling distribution, or merely Monte Carlo error? Which theoretical rate describes the sample mean's MSE? …

### Final diagnostic: change $B$ with $n$ fixed

We use a small prefix of an existing independent simulation and then the full simulation. This changes the number of replications while holding the data-generating process and sample size fixed.

In [ ]:
n_fixed = 12
x_big = draw_costs(B=12_000, n=n_fixed, mean_cost=MU0,
                   rng=np.random.default_rng(5555))
rows = []
for B_this in [500, 2_000, 12_000]:
    x = x_big[:B_this]
    means = x.mean(axis=1)
    se = x.std(axis=1, ddof=1) / np.sqrt(n_fixed)
    covered = is_covered(means, se, MU0,
                         stats.t.ppf(1-ALPHA/2, df=n_fixed-1))
    rows.append({
        "B": B_this,
        "MC bias (€)": mc_summary(means, MU0)["bias"],
        "MCSE bias (€)": mc_summary(means, MU0)["mcse_bias"],
        "coverage": covered.mean(),
        "MCSE coverage": coverage_summary(covered)["mcse_coverage"],
    })
display(pd.DataFrame(rows).set_index("B").round(4))

**Final question (2–3 sentences):** If you increase $B$ from 500 to 12,000 but keep $n=12$, does the *true coverage* of the $t$ interval change? What improves? …

---

### Submit

Submit your repaired notebook with all four checkpoints reading **`True`**, along with your written interpretations. Your reasoning about *why* a procedure works or fails is more important than matching a particular Monte Carlo decimal value.